# Task 3 — On-Device Efficiency Benchmarking and Quantization Trade-offs

**Owner:** Ali Cherri  
**Hypothesis (H3):** W4A8 quantization will cut end-to-end latency by ~35% with less than 3 pp accuracy loss.

**Variations:** ShowUI-2B, OS-Atlas-4B, Ferret-UI Lite-3B, best Qwen2.5-VL-3B (LoRA), best PaliGemma-3B (LoRA), each at fp16 / bnb-int8 / bnb-4bit (NF4).

**Deeper analysis:** profiling time spent in vision encoding vs. decoding; compression robustness (which models degrade most under low-bit inference, and on which benchmark).

In [ ]:
# Colab bootstrap. On a local Jupyter, this is a no-op since the env is already set up.
import sys
from pathlib import Path

REPO_URL = "https://github.com/ali-epita/action-learning-ais5.git"
REPO_DIR = "/content/action-learning-ais5"

if "google.colab" in sys.modules:
    if not Path(REPO_DIR).exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    # --no-deps keeps Colab's pre-bundled torch + CUDA build untouched.
    !pip install -q -e . --no-deps

    # transformers<5 because the Qwen2.5-VL wrapper targets 4.49+ and Colab now
    # ships 5.0. bitsandbytes is what the bnb8 / bnb4 quant cells need.
    !pip install -q "transformers>=4.49,<5" "accelerate>=0.34" "datasets>=3.0" \
                    "peft>=0.13" "qwen-vl-utils>=0.0.10" "bitsandbytes>=0.44" \
                    rich tqdm pyyaml typer pillow huggingface_hub safetensors psutil

    # pip editable install drops a .pth file site-packages doesn't rescan in
    # the current kernel — prepend src/ explicitly.
    src_dir = str(Path(REPO_DIR) / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import importlib
    importlib.invalidate_caches()

import ais5
print(f"ais5 v{ais5.__version__} ready  (colab={'google.colab' in sys.modules})")

In [ ]:
import pandas as pd

from ais5.bench import run_full_benchmark, pareto_front, ParetoPoint
from ais5.bench.memory import vram_budget_check
from ais5.data import load_benchmark
from ais5.models import get_model
from ais5.quant import resolve_quant_config
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)

In [ ]:
# Free Colab/Kaggle GPUs are 16GB; we enforce the 8GB on-device budget by
# measuring peak_vram_gb and rejecting cells that exceed it.
import torch
if torch.cuda.is_available():
    # Optional: cap VRAM at 8GB to make the budget binding rather than advisory.
    # torch.cuda.set_per_process_memory_fraction(8.0 / torch.cuda.get_device_properties(0).total_memory * (1024**3))
    print(torch.cuda.get_device_properties(0))

In [ ]:
MODELS = ['qwen2.5-vl-3b', 'paligemma-3b', 'showui-2b', 'os-atlas-4b']
QUANTS = ['none', 'bnb8', 'bnb4']
BENCHES = ['screenspot-v2', 'screenspot-pro', 'osworld-g']
LIMIT = 50  # set to None for the full benchmark

In [ ]:
import json
from pathlib import Path

# Cache samples per benchmark — without this we'd reload each set 12 times
# (once per model x quant). Big speedup once datasets are downloaded.
samples_by_bench = {b: list(load_benchmark(b)) for b in BENCHES}

results_dir = Path("results/bench")
results_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = results_dir / "task3_grid.jsonl"
jsonl_path.unlink(missing_ok=True)

rows = []
for model_name in MODELS:
    for quant_spec in QUANTS:
        qc = resolve_quant_config(quant_spec)
        kwargs = {"device_map": "auto"}
        hf_quant = qc.to_hf()
        if hf_quant is not None:
            kwargs["quant_config"] = hf_quant
        try:
            model = get_model(model_name, **kwargs)
        except Exception as e:
            print(f"SKIP {model_name}|{qc.name}: {e}")
            continue
        for bench in BENCHES:
            r = run_full_benchmark(
                model,
                samples_by_bench[bench],
                benchmark=bench,
                quant_label=qc.name,
                measure_components=True,
                limit=LIMIT,
            )
            row = r.to_dict()
            rows.append(row)
            with jsonl_path.open("a") as f:
                f.write(json.dumps(row) + "\n")
            print(
                f"{model_name}|{qc.name}|{bench}: "
                f"acc={row['accuracy']:.3f}  "
                f"lat={row['latency_mean_ms']:.0f}ms  "
                f"vram={row['peak_vram_gb']:.2f}GB"
            )
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(results_dir / "task3_grid.csv", index=False)
df.to_json(results_dir / "task3_grid.json", orient="records", indent=2)
df

In [ ]:
# Encoder-vs-decoder breakdown (Task 3's deeper-analysis component).
# Stacked bar per (model|quant) on one benchmark — the breakdown is roughly
# benchmark-independent within a model+quant cell, so V2 is sufficient.
import matplotlib.pyplot as plt

split_bench = "screenspot-v2"
rows_for_split = [
    r for r in rows
    if r["benchmark"] == split_bench and r.get("components_n")
]

if not rows_for_split:
    print(f"No component timings for {split_bench} — was measure_components=True?")
else:
    labels = [f"{r['model']}\n{r['quant']}" for r in rows_for_split]
    enc = [r["components_visual_encode_mean_ms"] for r in rows_for_split]
    dec = [r["components_llm_decode_mean_ms"] for r in rows_for_split]

    fig, ax = plt.subplots(figsize=(max(6.0, 0.9 * len(labels)), 4.5))
    ax.bar(labels, enc, label="visual encode", color="steelblue")
    ax.bar(labels, dec, bottom=enc, label="LLM decode + rest", color="salmon")
    ax.set_ylabel("mean ms / sample")
    ax.set_title(f"Per-step latency breakdown — {split_bench}")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    # Sortable table — encoder-bound vs decoder-bound configs.
    split = pd.DataFrame([
        {
            "model": r["model"],
            "quant": r["quant"],
            "visual_encode_ms": r["components_visual_encode_mean_ms"],
            "llm_decode_ms": r["components_llm_decode_mean_ms"],
            "visual_share_pct": r["components_visual_encode_share"] * 100,
        }
        for r in rows_for_split
    ])
    split = split.sort_values("visual_share_pct", ascending=False)
    display(split)

## Pareto fronts

Two views: latency vs. accuracy and VRAM vs. accuracy. The non-dominated set tells us which (model, quant) combinations are worth deploying.

In [ ]:
import matplotlib.pyplot as plt

# Published reference accuracies for models with no public weights yet.
# Source: Ferret-UI Lite-3B, arXiv 2509.26539 (core paper #1) — V2 91.6%, Pro 53.3%.
# Shown as horizontal dashed lines since we can't measure their latency/VRAM
# under the same conditions.
REFERENCE_POINTS = {
    "ferret-ui-lite-3b": {
        "screenspot-v2": 0.916,
        "screenspot-pro": 0.533,
    },
}

cost_views = [
    ("latency_mean_ms", "mean latency (ms)"),
    ("peak_vram_gb", "peak VRAM (GB)"),
]
benches_in_grid = sorted({r["benchmark"] for r in rows})

for bench in benches_in_grid:
    bench_rows = [r for r in rows if r["benchmark"] == bench]
    if not bench_rows:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, (cost_col, cost_label) in zip(axes, cost_views):
        pts = [
            ParetoPoint(
                label=f"{r['model']}|{r['quant']}",
                accuracy=r["accuracy"],
                cost=r[cost_col],
            )
            for r in bench_rows
        ]
        front = pareto_front(pts)
        ax.scatter([p.cost for p in pts], [p.accuracy for p in pts],
                   alpha=0.4, label="all configs")
        ax.plot([p.cost for p in front], [p.accuracy for p in front],
                "r-o", label="Pareto front")
        for p in front:
            ax.annotate(p.label, (p.cost, p.accuracy), fontsize=7)
        # Reference accuracies (no measurable cost — horizontal line).
        for ref_name, ref_data in REFERENCE_POINTS.items():
            if bench in ref_data:
                ax.axhline(
                    ref_data[bench], color="gray", linestyle="--", alpha=0.6,
                    label=f"{ref_name} (paper)",
                )
        # 8 GB on-device budget — only meaningful on the VRAM panel.
        if cost_col == "peak_vram_gb":
            ax.axvline(8.0, color="orange", linestyle=":", alpha=0.7,
                       label="8 GB budget")
        ax.set_xlabel(cost_label)
        ax.set_ylabel("accuracy")
        ax.set_title(f"{cost_label} vs. accuracy — {bench}")
        ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()

## Compression robustness

How much does each model lose when going from fp16 → 8-bit → 4-bit? Drop > 3 pp is flagged as failing H3.

In [ ]:
pivot = df.pivot_table(index=['model','benchmark'], columns='quant', values='accuracy')
pivot['drop_4bit'] = pivot.get('fp16', 0) - pivot.get('bnb-4bit-nf4', 0)
pivot